# Rategoan — 1 Klik: Training 100M + 200M + Auto-Release (GPU T4 gratis, Colab)

Varian **satu-klik** dari `colab-train-gpu.ipynb`: unduh korpus kanonik
K1/K2/K3 (5,22 miliar token BPE resmi, sudah bersih + terverifikasi
SHA256 - lihat `raget-data/jsonl/external/korpus-manifest-total.json`)
langsung dari GitHub Release → tokenisasi → training. Preset untuk
sekali `Runtime > Run all` langsung melatih **100M lalu 200M** berurutan
dan **otomatis menyimpan** hasilnya — 100M ke commit git (branch
training), 200M ke GitHub Release (checkpoint ini >100MB, tidak boleh
masuk git per kebijakan repo) — tanpa perlu centang/ubah parameter apa
pun secara manual di tengah jalan.

## Syarat SEKALI SAJA sebelum benar-benar "satu klik"

GitHub tetap butuh kredensial untuk menyimpan hasil training — ini tidak
bisa dilewati oleh notebook mana pun (bukan keterbatasan Rategoan, tapi
keamanan GitHub). Supaya run berikutnya benar-benar tinggal 1 klik:

1. Buat GitHub Personal Access Token (scope `repo`) sekali di
   [github.com/settings/tokens](https://github.com/settings/tokens).
2. Di Colab: ikon kunci 🔑 di sidebar kiri → **Add new secret** → nama
   `GITHUB_TOKEN`, isi tokennya → aktifkan akses notebook ini.
3. Selesai. Sel 1 di bawah akan otomatis membaca secret ini setiap run —
   tidak perlu paste token lagi.

Setelah secret tersimpan: buka notebook ini di Colab, pastikan
**Runtime > Change runtime type > T4 GPU**, lalu **Runtime > Run all**.
Selesai — sisanya otomatis (termasuk auto-resume kalau sesi Colab mati
di tengah jalan: tinggal Run all lagi, checkpoint terakhir dimuat balik).

## Ke mana hasilnya tersimpan (namanya persis ini)

| Model | Ukuran file | Tersimpan sebagai | Lokasi |
|---|---|---|---|
| 100M | ≈ 78 MB (di bawah batas 100MB GitHub) | commit git biasa | branch `colab-gpu-training` — `raget/raget-data/neural/raget-neural-massive100m.safetensors` |
| 200M | ≈ 163 MB (di atas batas 100MB GitHub) | GitHub Release asset | tag Release **`checkpoint-200m`** — asset `raget-neural-massive200m.safetensors` |

Laporan training tiap model (perplexity per ronde, dst) ikut ter-commit di
`raget/raget-devlog/neural/training-report-massive100m-round8-colab-gpu.json`
dan `training-report-massive200m-round8-colab-gpu.json` (nama file memuat
"round8" apa adanya — nama internal skrip training, bukan penomoran ulang).

## Catatan jujur

- Korpus yang diunduh (K1/K2/K3) adalah rak kanonik RESMI proyek —
  SATU-SATUNYA yang dipakai training, bukan regenerasi dari dump mentah
  seperti versi notebook sebelumnya. Setiap asset diverifikasi SHA256
  dulu terhadap digest Release sebelum dipakai (lihat sel unduh di
  bawah) - kalau tidak cocok, sel berhenti dengan error, bukan lanjut
  diam-diam dengan data yang mungkin korup/berubah.
- Total korpus kanonik saat ini (K1+K2+K3) = **5.223.639.069 token BPE
  resmi** dari 25.342.107 dokumen — lihat
  `raget-data/jsonl/external/korpus-manifest-total.json` untuk rincian
  per rak dan riwayat rekonsiliasi. Jauh melewati gerbang 1 miliar.
- Token/detik GPU T4 sungguhan HANYA diketahui setelah sel training
  selesai jalan - lihat `training-report-massive{50m,100m,200m}-round8-colab-gpu.json`
  untuk angka nyata, bukan estimasi notebook ini.
- fp16 (bukan bf16) dipakai karena GPU T4 = arsitektur Turing, tidak punya tensor core bf16 yang baik seperti Ampere+.
- MODE TUNTAS berhenti di epoch penuh ATAU target perplexity ATAU budget habis - budget habis BUKAN berarti training selesai, tinggal Run all lagi untuk lanjut (checkpoint+step diakumulasi otomatis dari metadata checkpoint sendiri).
- Checkpoint hasil Colab di-push ke branch terpisah (`colab-gpu-training` secara default) supaya tidak bentrok dengan commit yang sedang berjalan di sandbox lain - perlu di-merge manual (PR) atau oleh sesi Rategoan berikutnya.

In [ ]:
# @title 0. Cek GPU tersedia
import subprocess
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)
import torch
print('torch:', torch.__version__)
print('CUDA tersedia:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('PERINGATAN: tidak ada GPU terdeteksi - pastikan Runtime > Change runtime type > T4 GPU sudah dipilih (langkah manual 1).')

In [ ]:
# @title 1. Ambil GitHub token (Secret Colab atau input manual)
import os
GITHUB_TOKEN = None
try:
    from google.colab import userdata
    GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
except Exception:
    GITHUB_TOKEN = None
if not GITHUB_TOKEN:
    import getpass
    GITHUB_TOKEN = getpass.getpass('Secret GITHUB_TOKEN tidak ditemukan - paste GitHub PAT (scope repo) di sini: ')
assert GITHUB_TOKEN, 'Token GitHub wajib diisi untuk clone repo private + unduh aset Release + push hasil.'
os.environ['GITHUB_TOKEN'] = GITHUB_TOKEN
print('Token diterima (', len(GITHUB_TOKEN), 'karakter).')

In [ ]:
# @title 2. Clone repo Rategoan
REPO_OWNER = 'maetalizer-png'
REPO_NAME = 'Rategoan'
BRANCH = 'main'  # @param {type:"string"}

clone_url = 'https://' + GITHUB_TOKEN + '@github.com/' + REPO_OWNER + '/' + REPO_NAME + '.git'
!rm -rf /content/repo
!git clone --branch {BRANCH} --single-branch {clone_url} /content/repo
%cd /content/repo
!git config user.email 'colab-gpu-runner@rategoan.local'
!git config user.name 'Rategoan Colab GPU Runner'
!ls raget/raget-data/neural/
!ls raget/raget-tools/train-massive50m-colab-gpu.py

In [ ]:
# @title 3. Unduh K1/K2/K3 (korpus kanonik bersih, verifikasi SHA256 dulu) dari GitHub Release
import requests, os, gzip, shutil, hashlib

meta_headers = {'Authorization': 'Bearer ' + GITHUB_TOKEN, 'Accept': 'application/vnd.github+json'}
dl_headers = {'Authorization': 'Bearer ' + GITHUB_TOKEN, 'Accept': 'application/octet-stream'}

# (tag Release, nama asset, sha256 resmi dari korpus-manifest-total.json, path gz, path jsonl hasil ekstrak)
KANONIK_ASSETS = [
    ('korpus-ensiklopedia-bersih', 'korpus-ensiklopedia-bersih.jsonl.gz',
     '7ce4830db47ee6471a3cbe1234db41afcc779cbe307c0ec4c75163e10c125423',
     '/content/k1.jsonl.gz', '/content/k1.jsonl'),
    ('korpus-dialog-daerah-bersih', 'korpus-dialog-daerah-bersih.jsonl.gz',
     '3c64fee9bbe1927f99e05c7e0e29b655dc61668fdd56e7aab6f6ac0f4ecb704d',
     '/content/k2.jsonl.gz', '/content/k2.jsonl'),
    ('korpus-pelengkap-bersih', 'korpus-pelengkap-bersih.jsonl.gz',
     'ec07760453fa6cd9d3728ba5edc04b67f5c02f9c8061206060e5f0088ebf2d55',
     '/content/k3.jsonl.gz', '/content/k3.jsonl'),
]


def unduh_dan_verifikasi(tag, fname, sha_expected, gz_path, jsonl_path):
    r = requests.get('https://api.github.com/repos/' + REPO_OWNER + '/' + REPO_NAME + '/releases/tags/' + tag, headers=meta_headers)
    r.raise_for_status()
    release = r.json()
    asset = next((a for a in release['assets'] if a['name'] == fname), None)
    assert asset, 'Asset ' + fname + ' tidak ditemukan di Release ' + tag + ' - cek nama tag/asset masih sama di GitHub'
    print('Unduh', tag, '/', fname, '(', asset['size'], 'bytes)...')
    with requests.get('https://api.github.com/repos/' + REPO_OWNER + '/' + REPO_NAME + '/releases/assets/' + str(asset['id']), headers=dl_headers, stream=True, allow_redirects=True) as resp:
        resp.raise_for_status()
        with open(gz_path, 'wb') as f:
            for chunk in resp.iter_content(chunk_size=8 * 1024 * 1024):
                f.write(chunk)
    actual = os.path.getsize(gz_path)
    print('  selesai unduh:', actual, 'bytes (harus =', asset['size'], ')')
    assert actual == asset['size'], 'UKURAN TIDAK COCOK untuk ' + fname + ' - unduhan kemungkinan terpotong, ulangi sel ini'
    sha_actual = hashlib.sha256(open(gz_path, 'rb').read()).hexdigest()
    assert sha_actual == sha_expected, 'SHA256 TIDAK COCOK untuk ' + fname + ': dapat ' + sha_actual + ', harus ' + sha_expected + ' - JANGAN lanjut, data mungkin berubah/korup'
    print('  SHA256 cocok:', sha_actual)
    with gzip.open(gz_path, 'rb') as f_in, open(jsonl_path, 'wb') as f_out:
        shutil.copyfileobj(f_in, f_out)
    print('  diekstrak ke', jsonl_path, '(', os.path.getsize(jsonl_path), 'bytes)')


for tag, fname, sha, gz_path, jsonl_path in KANONIK_ASSETS:
    unduh_dan_verifikasi(tag, fname, sha, gz_path, jsonl_path)


In [ ]:
# @title 4. Ekstrak tokenizer (merges+vocab) LANGSUNG dari metadata checkpoint
# Token ID WAJIB identik dengan yang dipakai waktu checkpoint ini dilatih -
# bpe-tokenizer.json terpisah TIDAK dikomit ke git (lihat wikipedia-korpus-manifest.json),
# tapi merges+vocabEntries-nya tersimpan ulang di metadata checkpoint sendiri.
!python3 raget/raget-tools/extract-tokenizer-from-checkpoint.py raget/raget-data/neural/raget-neural-massive50m.safetensors /content/bpe-tokenizer.json
!ls -la /content/bpe-tokenizer.json

In [ ]:
# @title 5. Tokenisasi + chunking (window 512) K1/K2/K3, memakai tokenizer checkpoint
!python3 raget/raget-tools/tokenize-chunk-corpus.py /content/k1.jsonl /content/bpe-tokenizer.json /content/k1-chunks-512.txt 512
!python3 raget/raget-tools/tokenize-chunk-corpus.py /content/k2.jsonl /content/bpe-tokenizer.json /content/k2-chunks-512.txt 512
!python3 raget/raget-tools/tokenize-chunk-corpus.py /content/k3.jsonl /content/bpe-tokenizer.json /content/k3-chunks-512.txt 512


In [ ]:
# @title 6. Rechunk ke window 126 (sama seperti round 3/4/5 CPU - attention lebih murah) + gabung K1+K2+K3
!python3 raget/raget-tools/rechunk-corpus-window.py /content/k1-chunks-512.txt /content/k1-chunks-126.txt 126
!python3 raget/raget-tools/rechunk-corpus-window.py /content/k2-chunks-512.txt /content/k2-chunks-126.txt 126
!python3 raget/raget-tools/rechunk-corpus-window.py /content/k3-chunks-512.txt /content/k3-chunks-126.txt 126

!cat /content/k1-chunks-126.txt /content/k2-chunks-126.txt /content/k3-chunks-126.txt > /content/korpus-gabungan-126.txt
!wc -l /content/korpus-gabungan-126.txt


In [ ]:
# @title 7. TRAINING GPU - MODE TUNTAS, antrean 50m/100m/200m, auto-resume+auto-commit
MODEL_QUEUE = ['100m', '200m']  # @param {type:"raw"}
BUDGET_MINUTES_PER_MODEL = 300  # @param {type:"number"}
BATCH_SIZE = 96  # @param {type:"number"}
TARGET_PERPLEXITY = 0  # @param {type:"number"}
COMMIT_EVERY_MINUTES = 15  # @param {type:"number"}
PUSH_BRANCH = 'colab-gpu-training'  # @param {type:"string"}
CORPUS_FILE = '/content/korpus-gabungan-126.txt'  # gabungan K1+K2+K3 kanonik

for model_size in MODEL_QUEUE:
    print('\n\n========== TRAINING', model_size, '==========')
    !python3 raget/raget-tools/train-massive-colab-gpu.py {model_size} {BUDGET_MINUTES_PER_MODEL} {BATCH_SIZE} {CORPUS_FILE} {TARGET_PERPLEXITY} {COMMIT_EVERY_MINUTES} {PUSH_BRANCH}

print('\nAntrean selesai:', MODEL_QUEUE)
print('Sesi mati di tengah jalan? Klik Run all lagi - checkpoint terakhir (auto-commit tiap', COMMIT_EVERY_MINUTES, 'menit) dimuat balik otomatis, training lanjut, nol progres hilang.')

In [ ]:
# @title 8. Publikasikan checkpoint >100MB ke Release (OTOMATIS - 1 klik)
# Kebijakan Gudang Besar: checkpoint >100MB (200m ke atas) TIDAK boleh
# di-git-add (sudah otomatis dicegah git_auto_commit()) - harus diunggah
# sebagai Release asset dari sini (PAT Colab, bukan sandbox).
# Default True di notebook 1-klik ini - tidak perlu centang manual.
ARSIPKAN_CHECKPOINT_BESAR = True  # @param {type:"boolean"}
if ARSIPKAN_CHECKPOINT_BESAR:
    import os
    for size in ['200m', '300m', '500m']:
        p = 'raget/raget-data/neural/raget-neural-massive{}.safetensors'.format(size)
        if os.path.exists(p) and os.path.getsize(p) > 95 * 1024 * 1024:
            !python3 raget/raget-tools/publish-checkpoint-release.py {p} checkpoint-{size}
else:
    print('Dilewati (ARSIPKAN_CHECKPOINT_BESAR=False). Centang param di atas untuk mempublikasikan.')

In [ ]:
# @title 9. Commit + push final (jaring pengaman - skrip training sudah auto-commit tiap interval)
for model_size in MODEL_QUEUE:
    ckpt = 'raget/raget-data/neural/raget-neural-massive{}.safetensors'.format(model_size)
    report = 'raget/raget-devlog/neural/training-report-massive{}-round8-colab-gpu.json'.format(model_size)
    !git add {ckpt} {report}
!git commit -m "Round 8: sesi Colab selesai - queue {MODEL_QUEUE}"
!git push origin HEAD:{PUSH_BRANCH}
print('Selesai. Semua checkpoint di MODEL_QUEUE sudah di-push ke branch:', PUSH_BRANCH)
print('Merge branch ini ke main lewat Pull Request di GitHub, atau minta sesi Rategoan berikutnya untuk merge otomatis.')

In [ ]:
# @title 10. Ringkasan akhir - nama file & lokasi persis hasil training
import os
print('=== SELESAI - ringkasan lokasi hasil ===')
for size in MODEL_QUEUE:
    ckpt = 'raget/raget-data/neural/raget-neural-massive{}.safetensors'.format(size)
    report = 'raget/raget-devlog/neural/training-report-massive{}-round8-colab-gpu.json'.format(size)
    ada = os.path.exists(ckpt)
    mb = round(os.path.getsize(ckpt) / 1024 / 1024, 1) if ada else 0
    lokasi = 'GitHub Release tag checkpoint-{}'.format(size) if mb > 95 else 'commit git branch {}'.format(PUSH_BRANCH)
    print('-', size, ':', ckpt, '(' + str(mb) + ' MB)', '->', lokasi if ada else 'BELUM ADA - training belum selesai/gagal')
    print('  laporan:', report, '(ada)' if os.path.exists(report) else '(belum ada)')


## Catatan jujur

- Sel 3 mengunduh K1/K2/K3 LANGSUNG dari GitHub Release (bukan regenerasi dari dump mentah) - setiap asset diverifikasi ukuran + SHA256 dulu terhadap `korpus-manifest-total.json` sebelum dipakai, supaya kalau ada yang berubah/korup, sel berhenti dengan error jelas alih-alih lanjut diam-diam.
- Total korpus gabungan (K1+K2+K3) = 5.223.639.069 token BPE resmi dari 25.342.107 dokumen - lihat `raget-data/jsonl/external/korpus-manifest-total.json` untuk rincian per rak. Jauh di atas target jangka panjang 4 miliar.
- Token/detik GPU T4 sungguhan HANYA diketahui setelah sel 12 selesai jalan - lihat `training-report-massive{50m,100m,200m}-round8-colab-gpu.json` untuk angka nyata, bukan estimasi notebook ini.
- fp16 (bukan bf16) dipakai karena GPU T4 = arsitektur Turing, tidak punya tensor core bf16 yang baik seperti Ampere+.
- MODE TUNTAS berhenti di epoch penuh ATAU target perplexity ATAU budget habis - budget habis BUKAN berarti training selesai, tinggal Run all lagi untuk lanjut (checkpoint+step diakumulasi otomatis dari metadata checkpoint sendiri).
- Checkpoint hasil Colab di-push ke branch terpisah (`colab-gpu-training` secara default) supaya tidak bentrok dengan commit yang sedang berjalan di sandbox lain - perlu di-merge manual (PR) atau oleh sesi Rategoan berikutnya.

## Kebijakan Gudang Besar (Round 9, masih berlaku)

- **Rak korpus = barang bersih.** K1/K2/K3 di atas SATU-SATUNYA rak resmi - raw/staging tidak diarsipkan permanen (lihat `korpus-manifest-total.json` bagian `staging` untuk riwayat data yang di-RETIRE).
- **Rak otak = checkpoint <100MB saja di git.** Checkpoint >100MB (batas keras GitHub, saat ini `massive200m` di 163,41MB) TIDAK di-git-add (`git_auto_commit()` sudah otomatis skip) - sel 13b (opsional) mempublikasikannya ke Release tag `checkpoint-<ukuran>`. TIDAK pakai Git LFS (kuota bandwidth gratis GitHub gampang habis).
- **Kenapa sel 13b harus dijalankan dari Colab, bukan sandbox:** sesi sandbox Claude Code Remote (tempat kode ini biasanya dikembangkan) diblokir dari operasi Release (`"Creating, editing, or deleting releases is not permitted for this session type"`) walau boleh membaca/mengunduh - jadi publikasi Release HARUS lewat sini (token Colab) atau mesin lokal. Detail: `raget-tools/CHECKPOINT-POLICY.md`.
